# Notebook 02 — Experimental Structure Determination

**Module:** 11 — Structural Biology  
**Tier:** 3 — Survey  
**Notebook:** 02 of 08  
**Time estimate:** 60 minutes

> Before you can compute on a protein structure, someone had to determine it experimentally.
> Understanding how structures are obtained — and their limitations — is essential
> for interpreting the PDB entries you will use computationally.

---
## Step 1 — Motivation

The PDB contains >215,000 structures, but they were obtained by different methods
with different resolutions, biases, and limitations. A computational biologist must
understand what method produced a structure to correctly interpret it:
- X-ray: rigid average; can't observe dynamics
- cryo-EM: can handle large complexes; resolution improving rapidly
- NMR: solution state; limited to small proteins; gives ensemble

---
## Step 2 — Intuition

**X-ray crystallography:** Pack millions of identical protein molecules into a crystal.
Shine X-rays through. The crystal acts as a diffraction grating — X-rays scatter off
the repeating electron density and produce a diffraction pattern. Solve mathematically
to recover the electron density map, then fit atoms to it.

**Cryo-EM:** Freeze a solution of proteins in vitreous ice (so fast they can't
crystallize). Take 100,000+ electron microscope images. Each image shows one protein
at a random orientation. Use single-particle analysis to reconstruct the 3D structure
from these random projections.

**NMR:** Spin protein nuclei (¹H, ¹³C, ¹⁵N) in a strong magnetic field. Measure the
resonance frequencies — they are sensitive to the local chemical environment and
distances between nearby nuclei. Collect thousands of distance constraints and compute
an ensemble of structures consistent with the data.

---
## Step 3 — Biological Background

### X-ray Crystallography

**Workflow:**  
1. Purify protein → grow crystal (often the hardest step)
2. Collect diffraction data at a synchrotron X-ray source
3. Solve the **phase problem** (observed intensities, not phases): use molecular
   replacement (if homologous structure exists) or experimental phasing (MAD/SAD)
4. Build atomic model into electron density map; refine

**Resolution:** Reported in Ångströms. <2.0 Å = high resolution (side chains clear);
2.0–3.0 Å = medium (backbone and many side chains); >3.5 Å = low (backbone only).

**R-factor:** Measures agreement between observed and calculated structure factors.
$R_{\text{work}} < 0.25$ is typical; $R_{\text{free}} < 0.30$ validates against
data held out during refinement.

### Cryo-EM

**Resolution revolution (2013–present):** Introduction of direct electron detectors
improved cryo-EM resolution from ~10 Å to 1.5–3.0 Å routinely. Now the dominant
method for large complexes (ribosomes, membrane proteins, viruses).

**Key advantage:** No crystallization required. Can capture multiple conformational
states in one dataset (heterogeneous reconstruction).

**Limitation:** Still struggles with very small proteins (<50 kDa), flexible regions.

### NMR

**Output:** An ensemble of 10–20 structures all consistent with the experimental
constraints. RMSD between ensemble members reflects local flexibility.

**Limitation:** Practical upper limit ~50 kDa; more difficult for membrane proteins
and aggregation-prone proteins.

**Advantage:** Gives dynamics information; solution state (no crystal packing artifacts).

---
## Step 4 — Mathematical Explanation

**X-ray structure factor:**
$$F(\mathbf{h}) = \sum_j f_j \exp(2\pi i \mathbf{h} \cdot \mathbf{r}_j)$$

where $f_j$ = atomic scattering factor, $\mathbf{r}_j$ = atom position, $\mathbf{h}$ =
reciprocal lattice vector (Miller indices h,k,l).

The **electron density** is the inverse Fourier transform:
$$\rho(\mathbf{r}) = \frac{1}{V} \sum_{\mathbf{h}} F(\mathbf{h}) \exp(-2\pi i \mathbf{h} \cdot \mathbf{r})$$

Diffraction gives $|F(\mathbf{h})|^2$ (intensities) but not phases — this is the
**phase problem**. Without phases, the inverse Fourier transform cannot be computed.

**Cryo-EM reconstruction (Central Projection Theorem):**  
A 2D projection of a 3D object at orientation $(\theta, \phi)$ corresponds to a
central slice through the 3D Fourier transform. Collecting projections at all
orientations fills the 3D Fourier space; inverse FT gives the 3D density.

**B-factor (Debye-Waller factor):**
$$f_j(\text{with disorder}) = f_j \exp\!\left(-B_j \frac{\sin^2\theta}{\lambda^2}\right)$$

$B_j$ reflects atomic displacement (thermal motion + static disorder). High B-factors
(>60 Å²) indicate flexible regions; low B-factors (<10 Å²) indicate rigid regions.

In [ ]:
# Step 6 — Python: Simulate PDB deposition statistics and method comparison
import numpy as np
import matplotlib.pyplot as plt

# ---- Approximate PDB deposition data (public knowledge, rounded) ----
# Source: RCSB PDB statistics (June 2026)
years = np.arange(1976, 2027)

# Cumulative depositions by method (rough approximation for illustration)
# X-ray dominates until ~2015, then cryo-EM rises sharply
xray_annual = np.array([
    1, 2, 4, 5, 7, 8, 10, 12, 18, 25, 35, 50, 70, 100, 130, 160,    # 1976–1991
    200, 250, 300, 400, 500, 700, 900, 1100, 1300, 1600, 1900, 2200, # 1992–1999
    3000, 3800, 4500, 5200, 6000, 6800, 7500, 8000, 8500, 9000,      # 2000–2007
    9500, 10000, 10500, 10800, 11000, 11200, 11000, 10500, 10000,     # 2008–2016
    9500, 9000, 8500,                                                  # 2017–2019
    9000, 9500, 10000,                                                 # 2020–2022
    10000, 10500, 11000, 11000                                         # 2023–2026
])
cryoem_annual = np.array([
    *[0]*30,  # 1976–2005: negligible
    5, 10, 20, 40, 100, 200, 500, 1000, 1500,  # 2006–2014: slow rise
    2000, 3000, 4500, 6000, 7500,              # 2015–2019: revolution
    9000, 11000, 13000, 14000, 15000, 16000, 16000  # 2020–2026
])
nmr_annual = np.array([
    *[1]*10,
    5, 10, 20, 40, 100, 200, 400, 700, 900, 1100, 1300, 1500, 1600,  # 1986–1998
    1700, 1800, 1900, 2000, 2100, 2200, 2300, 2400, 2500, 2600, 2700,
    2800, 2900, 3000, 3100, 3200, 3300, 3400, 3500, 3600, 3700, 3800,
    3900, 4000, 4000, 4000, 4000, 4000, 4000, 4000
])

# Ensure arrays are same length
n = len(years)
xray_annual   = xray_annual[:n]
cryoem_annual = cryoem_annual[:n]
nmr_annual    = nmr_annual[:n]

# Cumulative
xray_cum   = np.cumsum(xray_annual)
cryoem_cum = np.cumsum(cryoem_annual)
nmr_cum    = np.cumsum(nmr_annual)
total_cum  = xray_cum + cryoem_cum + nmr_cum

print(f'Approximate PDB statistics (June 2026):')
print(f'  X-ray structures:   ~{xray_cum[-1]:,}')
print(f'  Cryo-EM structures: ~{cryoem_cum[-1]:,}')
print(f'  NMR structures:     ~{nmr_cum[-1]:,}')
print(f'  Total:              ~{total_cum[-1]:,}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel A: PDB growth over time
ax = axes[0]
ax.stackplot(years, xray_annual, cryoem_annual, nmr_annual,
              labels=['X-ray', 'Cryo-EM', 'NMR'],
              colors=['steelblue', 'tomato', 'forestgreen'], alpha=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('Structures deposited per year')
ax.set_title('A. PDB depositions per year\n(approximate, by method)')
ax.legend(fontsize=8)
ax.set_xlim(1990, 2027)

# Panel B: Comparison table (text panel)
ax = axes[1]
ax.axis('off')
cols = ['Method', 'Resolution', 'Size limit', 'Crystallization?', 'Dynamics?']
rows = [
    ['X-ray', '1–3.5 Å', 'Any if crystal', 'Required', 'No (static average)'],
    ['Cryo-EM', '1.5–5 Å', '>150 kDa ideal', 'Not required', 'Multiple states'],
    ['NMR', '~2–4 Å equiv.', '<50 kDa', 'Not required', 'Yes (ensemble)'],
    ['AlphaFold2', 'N/A (predicted)', 'Any sequence', 'Not required', 'No'],
]
table = ax.table(cellText=rows, colLabels=cols, cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(7)
table.scale(1, 1.8)
ax.set_title('B. Method comparison')

# Panel C: Resolution histogram concept (B-factor effect)
ax = axes[2]
resolutions_common = np.concatenate([
    np.random.default_rng(42).normal(2.2, 0.5, 200),  # X-ray
    np.random.default_rng(43).normal(3.0, 0.7, 80),   # cryo-EM
])
resolutions_common = np.clip(resolutions_common, 1.0, 6.0)
ax.hist(resolutions_common, bins=40, color='steelblue', alpha=0.8, edgecolor='none')
ax.axvline(2.0, color='forestgreen', ls='--', label='High res (<2 Å)')
ax.axvline(3.5, color='tomato', ls='--', label='Low res (>3.5 Å)')
ax.set_xlabel('Resolution (Å) — lower is better')
ax.set_ylabel('Count')
ax.set_title('C. Typical resolution distribution\n(simulated, X-ray + cryo-EM)')
ax.legend(fontsize=8)

plt.suptitle('Module 11 NB02: Experimental Structure Determination', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('structure_determination.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. A PDB entry has resolution 4.5 Å and R-free 0.38. What does this tell you about
   the reliability of side-chain positions?
2. Why can cryo-EM determine structures of membrane proteins more easily than X-ray
   crystallography?
3. What is the phase problem in X-ray crystallography and how is molecular replacement
   used to solve it?
4. A protein NMR ensemble has 20 models with average Cα RMSD of 0.3 Å for the core
   and 3.5 Å for the C-terminal tail. What does this tell you?

---
## Step 10 — Quiz

1. What is the B-factor and what does a high B-factor indicate?
2. What is the Central Projection Theorem and why is it relevant to cryo-EM?
3. Why did cryo-EM resolution improve dramatically after 2013?
4. What is the phase problem and why does it make X-ray structure determination hard?
5. Which method gives information about protein dynamics in solution?

---
## Step 12 — Reflection

> *[In one paragraph: you have a 250 kDa membrane protein complex and want to
> determine its structure. Explain which experimental method you would choose
> and why the alternatives are less suitable.]*

---
*Next: `03_pdb_format_parsing.ipynb`*